# LiH ground state with VQE
PySCF + qiskit-nature | Jordan-Wigner + UCCSD | frozen core (10 qubits)

Install: `pip install "qiskit<2" "qiskit-nature>=0.7.2" qiskit-algorithms pyscf`

In [ ]:
import numpy as np
from qiskit.primitives import Estimator
from qiskit_algorithms import NumPyMinimumEigensolver, VQE
from qiskit_algorithms.optimizers import SLSQP
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.algorithms import GroundStateEigensolver
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.transformers import FreezeCoreTransformer

## 1. Molecule (PySCF) + frozen core
STO-3G LiH has 6 spatial orbitals (12 qubits). Freezing the Li 1s core leaves 5 -> **10 qubits**.

In [ ]:
driver = PySCFDriver(
    atom="Li 0 0 0; H 0 0 1.595",
    basis="sto3g",
    charge=0,
    spin=0,
    unit=DistanceUnit.ANGSTROM,
)
problem = FreezeCoreTransformer().transform(driver.run())

print("spatial orbitals:", problem.num_spatial_orbitals)
print("particles (a, b):", problem.num_particles)

## 2. Jordan-Wigner mapping + UCCSD ansatz (Hartree-Fock start)

In [ ]:
mapper = JordanWignerMapper()

hf_state = HartreeFock(problem.num_spatial_orbitals, problem.num_particles, mapper)
ansatz = UCCSD(
    problem.num_spatial_orbitals,
    problem.num_particles,
    mapper,
    initial_state=hf_state,
)
print("qubits:", ansatz.num_qubits, "| parameters:", ansatz.num_parameters)

## 3. Run VQE

In [ ]:
vqe = VQE(
    Estimator(),
    ansatz,
    SLSQP(maxiter=200),
    initial_point=np.zeros(ansatz.num_parameters),  # start at Hartree-Fock
)
vqe_result = GroundStateEigensolver(mapper, vqe).solve(problem)
e_vqe = vqe_result.total_energies[0].real

## 4. Exact reference + comparison

In [ ]:
exact_result = GroundStateEigensolver(mapper, NumPyMinimumEigensolver()).solve(problem)
e_exact = exact_result.total_energies[0].real

CHEM_ACC = 1.6e-3  # Ha
print(f"VQE   : {e_vqe:.6f} Ha")
print(f"Exact : {e_exact:.6f} Ha  (target ~ -7.882176)")
print(f"Error : {abs(e_vqe - e_exact):.2e} Ha")
print("Within chemical accuracy:", abs(e_vqe - e_exact) < CHEM_ACC)